# Tutorial: How to Read `.bag` Data

In [19]:
import pyrealsense2 as rs
import os
import numpy as np
import cv2
import gc

## Definitions

When you attempt to record videos using an Intel Realsense camera (e.g. D435i RGB-D camera, L515 LiDAR camera), the output of that operation is usually a `.bag` file. Realsense doesn't distinguish between cameras and `.bag` files; they are, in Intel's eyes, **essentially the same thing**. So consider your `.bag` files as "instances" of your camera.

This `.bag` data contains one or several "streams" of data. Each stream represents some element of the recording - i.e. the depth video, or the RGB video. In essence, a single `.bag` file can contain multiple streams.

Each stream in of itself has a specific type, format, resolution, FPS, etc. These are things you MUST note down when you are recording your footage. The `.bag` data doesn't contain any of this information, as far as I am aware.

## Goal of this Notebook

If we're given a `.bag` file to process, then it's expected that we want to extract a video per stream contained within that `.bag` file. We need to be _careful_ about this, as there are nuances to how `.bag` files work that cannot be ignored.

We have an example `.bag` file: `./samples_ignore/capstone/20251116_155458.bag`.

In [23]:
_EXAMPLE_BAG = './samples_ignore/capstone/20251116_155458.bag'

### Common "Configuration" Class

Let's use a common class we can use EVERYWHERE to define things such as the width and height of a stream (i.e. the resolution).

In [ ]:
class _CONFIG_:
    def __init__(self, name:str, w:int, h:int, fps:int):
        self.name = name
        self.width = w
        self.height = h
        self.fps = fps

# In the case of an Intel Realsense stream:
class _INTEL_CONFIG_(_CONFIG_):
    def __init__(self, name:str, w:int, h:int, fps:int, src:str, type:rs.stream, format:rs.format):
        super().__init__(name, w, h, fps)
        self.src = src
        self.type = type
        self.format = format

# In the case of an OpenCV-based video output:
class _OUTPUT_CONFIG_(_CONFIG_):
    def __init__(self, name:str, w:int, h:int, fps:int, interpolator=cv2.INTER_CUBIC, record:bool=True, record_filename:str=None, preview:bool=False):
        super().__init__(name, w, h, fps)
        self.interpolator = interpolator
        self.record = record
        self.preview = preview
        self.record_filename = name+".mp4" if record_filename is None else record_filename+".mp4"

        # if neither record or preview are set to True, will auto-default to record=true, preview=false
        if not self.record and not self.preview:
            print("ERROR: OUTPUT CONFIG CANNOT BE NEITHER RECORD OR PREVIEW. DEFAULTING TO RECORDING")
            self.record = True
            self.preview = False

Let's try to use these configs in practice. Our demo `.bag` file was filmed in 1280x720 aspect ratio with 30FPS. We want to output a smaller version of that: a 640x480 aspect ratio with 30FPS.

In [32]:
intel_config = _INTEL_CONFIG_('20251116_155458-Color', w=1280, h=720, fps=30, src=_EXAMPLE_BAG, type=rs.stream.color, format=rs.format.rgb8)
output_config = _OUTPUT_CONFIG_('20251116_155458-Color', w=640, h=480, fps=30, record=False, preview=True)

## Stream Readers and Writers

Remember that each `.bag` file has its own set of streams. So we need some way to read each stream and output each stream. We also need to find a way to write each stream to either an OpenCV window (for previewing) or recording output (for saving videos).

To achieve this, we'll create a single `Stream` class that handles all that stuff.

In [ ]:
from random import choice
from string import ascii_uppercase

class Stream:
    def __init__(self, 
                 input_bag:str,
                 intel_config:_INTEL_CONFIG_, 
                 output_config=_OUTPUT_CONFIG_):
        
        # Inputs
        self.bag = input_bag
        self.intel_config = intel_config
        self.output_config = output_config

        # Realsense pipelines
        pipeline = rs.pipeline()
        config = rs.config()
        profile = None

    # Starting a pipeline
    def enable_pipeline(self):
        self.profile = self.pipeline.start(self.config)

    # "stopping" a pipeline
    def disable_pipeline(self):
        self.pipeline.stop()
        self.profile = None

    # Helper: Make directories. Delete them if they exist already
    def mkdirs(self, query_dir:str, delete_existing:bool=True):
        # If the folder already exists, delete it
        if delete_existing and os.path.exists(query_dir): shutil.rmtree(query_dir)
        # Create a new empty directory
        os.makedirs(query_dir, exist_ok=True)
        # Return the directory to indicate completion
        return query_dir


    # Read a pipeline and extract all frames. We ensure to sort them
    def get_frames(self, frames_dir:str=None):
        # Determine a safe output folder to put all frame data in
        fdir = ''.join(choice(ascii_uppercase) for i in range(12)) if frames_dir is None else frames_dir
        self.frames_dir = self.mkdirs(fdir)

        

        

In [18]:
## VERY SIMPLE VIDEO WRITER CLASS, USING OPENCV-PYTHON
# This handles most issues with writing, insofar as handling `.bag` data is concerned.
# Use this if you expect that you will be writing to video outputs.

class VideoWriter:
    def __init__(self,
        filename:str,
        width:int,
        height:int,
        fps:int,
        interpolator=None):

        self.filename = filename
        self.width = width
        self.height = height
        self.fps = fps
        self.interpolator = interpolator
        self.video_writer = None

    def start(self):
        try:
            self.video_writer = cv2.VideoWriter(
                self.filename, 
                cv2.VideoWriter_fourcc(*'MP4V'), 
                self.fps, 
                [self.width, self.height]
            )
        except Exception as error:
            print(f"Unable to initialize output of {self.filename}: ", error)
    
    def add_frame(self, frame):
        if self.video_writer is None: return
        out_image = cv2.resize(frame, (self.width, self.height), self.interpolation)
        self.video_writer.write(out_image)
    
    def close_writer(self):
        if self.video_writer is not None:
            self.video_writer.release()
            self.video_writer = None

In [ ]:
class VideoPreview
    def __init__(self, )

## Step 1: Defining a Stream

An Intel Realsense camera consists of several "streams" that are recorded simultaneously. Think of an RGB video recorded simultaneously with a depth map video. Here, the `Stream` class follows the same conceptual idea.

In [ ]:
class Stream:
    def __init__(
        self, 
        stream_name:str,        # The name of this stream, as a unique identifier
        type:rs.stream,         # The type of stream (ex. rs.stream.color, .depth)
        format: rs.format,      # How the stream should be parsed (ex: rs.format.z16, .rgb8, .bgr8, .yuyv)
        width: int,             # The original stream's width (e.g. 1280, 640)
        height: int,            # The original stream's height (e.g. 720, 480)
        fps:int,                # The original stream's fps (e.g. 30, 15)
    ):
        self.name = stream_name
        self.type = type
        self.format = format
        self.width = width
        self.height = height
        self.fps = fps

        print(f"Intel Realsense Stream Initialized: \"{self.name}\"")
        print(f"\tTypes: {self.type} w/ {self.format}")
        print(f"\tInput: (w:{self.width}, h:{self.height}) @ {self.fps} FPS")

    def set_bag(self, src:str):
        self.bag = src if os.path.exists(src) else None

    def enable_stream(self, config):
        config.enable_stream(self.type, self.width, self.height, self.format, self.fps)

    def enable_writer(self, 
                      out_filename:str, 
                      out_width, 
                      out_height:int,
                      out_fps:int,
                      interpolator=None):
        try:
            self.video_writer = cv2.VideoWriter(
                out_filename, 
                cv2.VideoWriter_fourcc(*'MP4V'), 
                out_fps, 
                [out_width, out_height]
            )
        except Exception as error:
            print(f"Unable to initialize output of {self.out_filename}: ", error)
    
    def add_frame(self, frame):
        if self.video_writer is None: return
        out_image = cv2.resize(frame, (self.out_width, self.out_height), self.out_interpolation)
        self.video_writer.write(out_image)
    
    def close_writer(self):
        if self.video_writer is not None:
            self.video_writer.release()
            self.video_writer = None
    

In [5]:
ex_stream = Stream('RGB', rs.stream.color, rs.format.rgb8, 1280, 720, 30)

Intel Realsense Stream Initialized: "RGB"
	Types: stream.color w/ format.rgb8
	Input: (w:1280, h:720) @ 30 FPS


## Step 2: Setting Up Configs

With the `Stream` established, nowe we need to set up some configurations to read that stream.

In [9]:
pipeline = rs.pipeline()
config = rs.config()
profile = None

These three basically establish the core idea of reading a stream:

- The `pipeline` iterates through frames of a stream, giving you each frame.
- The `config` lets you control elements of the `pipeline`
- The `profile` is an instance of the `pipeline` running. We can use this to prevent multiple versions of the same pipeline from loading.

## Step 3: Loading a Device via the `.bag` File

Now, let's attempt to load in a `.bag` file. The Intel Realsense ecosystem interprets each `.bag` file as an instance of a device, which may hold multiple `Streams`. So let's call this stage "Loading a Device".

The example `.bag` file we want to mess with is under `samples_ignore/capstone/20251116_155458.bag`.

In [13]:
rs.config.enable_device_from_file(
    config, 
    "../samples_ignore/capstone/20251116_155458.bag", 
    repeat_playback=False)

## Step 4: Starting (and Stopping) a Pileline

Every time we run a pipeline to start and stop reading frames from a `.bag` file, let's call that entire session a "profile".

In [14]:
# How to "start" a pipeline
profile = pipeline.start(config)

# How to "stop" a pipeline
pipeline.stop()
profile = None

## Step 5: Reading Frames

This step is a bit complicated, so let's describe it like this:

1. Your `.bag` file is considered a device that is streaming frames. To read those frames, we've set up a `pipeline` that is controlled by a `config`uration.
2. When we read frames, because this is a `.bag` file, we have a finite number of frames. However, when we iterate throug frames, **that doesn't mean that all the frames will be captured on the first run**. In fact, there are usually cases where some frames are skipped for no reason.
3. To prevent the problem of skipped frames, we do two things:
    1. Extract the frame number of a given frame
    2. Sort all frames based on the frame numbers
    3. Re-build the frames in order, potentially even creating a video.